# 🎬 Reverse Prompt Engineering for AI Educational Videos
**向上溯源反推系统 - Google Drive 实时同步版**

本 Notebook 直接从 Google Drive 读取项目代码（无需 git clone）。
在本地修改代码后，Drive 自动同步，Colab 重新运行即可获得最新版本。

Pipeline: Mount Drive → Install Deps → Download Videos → Run Pipeline → Pattern Analysis

In [ ]:
# ============================================================
# CELL 1: Check GPU
# ============================================================
import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'
vram = f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else 'N/A'
print(f'GPU : {gpu_name}')
print(f'VRAM: {vram}')
if not torch.cuda.is_available():
    print('⚠️  请在 Runtime > Change runtime type 中选择 T4 GPU')

In [ ]:
# ============================================================
# CELL 2: Mount Google Drive & Set Project Path
# ============================================================
# 前提：本地已安装 Google Drive for Desktop，项目文件夹已同步到 Drive
# 只需修改下面的 PROJECT_DRIVE_PATH 为你的项目在 Drive 中的路径

from google.colab import drive
import os

drive.mount('/content/drive')

# ⚙️ 修改这里：Drive 中项目文件夹的路径（相对于 MyDrive）
PROJECT_SUBPATH = '工坊省级项目'  # 如果放在子文件夹里，例如 'competetion/工坊省级项目'

PROJECT_DIR = f'/content/drive/MyDrive/{PROJECT_SUBPATH}'

if not os.path.exists(PROJECT_DIR):
    raise FileNotFoundError(
        f'❌ 项目目录不存在: {PROJECT_DIR}\n'
        f'请确认:\n'
        f'  1. Google Drive for Desktop 已在本地安装并登录同一账号\n'
        f'  2. 本地项目文件夹已存放在 Google Drive 同步目录内\n'
        f'  3. PROJECT_SUBPATH 路径设置正确'
    )

os.chdir(PROJECT_DIR)
print(f'✅ Working directory: {os.getcwd()}')
print(f'Files: {os.listdir(".")}')

In [ ]:
# ============================================================
# CELL 3: Install Dependencies
# ============================================================
!pip install -r requirements.txt -q
print('✅ Dependencies installed')

In [ ]:
# ============================================================
# CELL 4: Cache HuggingFace Models to Google Drive
# 首次运行会下载 InternVL2-2B (~4GB) + Qwen2.5-1.5B (~3GB)
# 缓存到 Drive 后，后续 session 秒加载
# ============================================================
import os

HF_CACHE_DIR = '/content/drive/MyDrive/hf_cache'
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE_DIR
os.environ['TRANSFORMERS_CACHE'] = HF_CACHE_DIR
print(f'✅ HuggingFace 模型缓存目录: {HF_CACHE_DIR}')
print('   首次运行将自动下载模型权重到 Google Drive (~7GB 合计)')
print('   后续 session 将直接从 Drive 加载，无需重新下载')

In [ ]:
# ============================================================
# CELL 5: Download Video Dataset
# ============================================================
# golden: 高质量 AIGC 教学视频 (3Blue1Brown, Kurzgesagt 等)
# regular: 普通教学视频（讲义/白板/录屏风格）
# 视频下载到 Drive 项目目录内，永久保存，无需每次重下

import os
MAX_VIDEOS_PER_GROUP = 5  # 每组最多下载视频数

golden_dir = 'data/raw_videos/golden'
regular_dir = 'data/raw_videos/regular'
golden_count = len([f for f in os.listdir(golden_dir) if f.endswith('.mp4')]) if os.path.exists(golden_dir) else 0
regular_count = len([f for f in os.listdir(regular_dir) if f.endswith('.mp4')]) if os.path.exists(regular_dir) else 0

print(f'已有视频: Golden={golden_count}, Regular={regular_count}')

if golden_count < MAX_VIDEOS_PER_GROUP or regular_count < MAX_VIDEOS_PER_GROUP:
    !python main.py --download --max {MAX_VIDEOS_PER_GROUP}
    print('✅ 下载完成')
else:
    print('✅ 视频已存在，跳过下载')

In [ ]:
# ============================================================
# CELL 6: Run Pipeline - Golden Videos
# ============================================================
# 流程: 关键帧提取 → InternVL2-2B 视觉分析 → 3个Agent推理 → 提示词合成
!python main.py --batch data/raw_videos/golden/ --group golden --output results/golden/ --lang en

In [ ]:
# ============================================================
# CELL 7: Run Pipeline - Regular Videos
# ============================================================
!python main.py --batch data/raw_videos/regular/ --group regular --output results/regular/ --lang en

In [ ]:
# ============================================================
# CELL 8: Pattern Analysis
# ============================================================
import os, shutil

os.makedirs('results/all', exist_ok=True)
for src_dir in ['results/golden', 'results/regular']:
    if os.path.exists(src_dir):
        for f in os.listdir(src_dir):
            if f.endswith('.json'):
                shutil.copy(os.path.join(src_dir, f), 'results/all/')

!python main.py --analyze --results results/all/
print('✅ 分析完成，结果已保存到 Drive')

In [ ]:
# ============================================================
# CELL 9: View Results
# ============================================================
import json, os

report_path = 'results/all/pattern_report.json'
if os.path.exists(report_path):
    with open(report_path) as f:
        report = json.load(f)
    
    print('=== 🏆 Golden 提示词高频关键词 ===')
    print(', '.join(report.get('golden_keywords', [])[:15]))
    
    print('\n=== 📚 Regular 提示词高频关键词 ===')
    print(', '.join(report.get('regular_keywords', [])[:15]))
    
    print('\n=== 📊 统计数据 ===')
    print(json.dumps(report.get('stats', {}), indent=2))
    
    print('\n=== 💡 分析摘要 ===')
    print(report.get('summary', ''))
    
    print('\n=== 🥇 关键词优势对比（Golden vs Regular Top 10）===')
    freq = report.get('keyword_frequency', [])
    for item in freq[:10]:
        bar = '█' * int(item['golden_advantage'] * 10)
        print(f"  {item['keyword']:20s} Golden:{item['golden_pct']:5.1f}% Regular:{item['regular_pct']:5.1f}% +{bar}")
else:
    print('❌ 未找到报告，请先运行 Cell 8')

In [ ]:
# ============================================================
# CELL 10: GPU Memory Check
# ============================================================
import torch
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved() / 1e9
    total     = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU Memory: {allocated:.1f}GB allocated / {reserved:.1f}GB reserved / {total:.1f}GB total')
    print(f'Available : {total - reserved:.1f}GB')